# cellduet — environment smoke test

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PatrickJReed/cellduet/blob/main/notebooks/00_environment_smoke.ipynb)

Run this **first** in any Colab session to verify the runtime is wired up correctly. It checks:

1. GPU available (or graceful CPU fallback)
2. `cellduet` package installs from GitHub and imports
3. Hugging Face Hub login works (via Colab Secret `HF_TOKEN`)
4. Google Drive mounts (optional; for persistent caching)
5. HF read access works (lists your account)

If all five pass, the runtime is good to go for analysis notebooks.

First-time setup: see `docs/SETUP.md` in the repo.

## 1. GPU check

In [ ]:
!nvidia-smi || echo "No GPU allocated; runtime is CPU-only. Switch via Runtime > Change runtime type if needed."

## 2. Install cellduet from GitHub

Always pulls the latest `main`. Takes ~30-60 s on first install.

In [ ]:
!pip install -q git+https://github.com/PatrickJReed/cellduet.git@main

In [ ]:
import cellduet
print(f"cellduet {cellduet.__version__} imported OK")

## 3. Hugging Face login

Reads token from the Colab Secret `HF_TOKEN`. If you have not added it yet: left sidebar key icon -> + Add new secret -> name `HF_TOKEN`, value your `hf_...` write token, toggle Notebook access on.

Outside Colab, this falls back to the local `huggingface-cli login` cache.

In [ ]:
from huggingface_hub import login, HfApi

try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    login(token=token, add_to_git_credential=False)
    print("HF login: OK (via Colab secret)")
except Exception as e:
    # Outside Colab or no secret set: assume local cache exists
    print(f"Colab secret not used ({type(e).__name__}); relying on local huggingface-cli cache")

api = HfApi()
me = api.whoami()
print(f"HF authenticated as: {me['name']}")

## 4. Drive mount (optional, Colab only)

Persistent storage at `/content/drive/MyDrive/cellduet/` survives session resets. Skip if running outside Colab or if you don't need persistent caching this session.

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    cache_dir = '/content/drive/MyDrive/cellduet/cache'
    os.makedirs(cache_dir, exist_ok=True)
    print(f"Drive mounted; cache dir ready at {cache_dir}")
except (ImportError, Exception) as e:
    print(f"Drive mount skipped ({type(e).__name__}): {e}")

## 5. Smoke-test summary

In [ ]:
import sys
import platform

print("cellduet smoke test summary")
print("-" * 40)
print(f"Python:         {sys.version.split()[0]}")
print(f"Platform:       {platform.platform()}")
print(f"cellduet:       {cellduet.__version__}")
print(f"HF user:        {me['name']}")
print("")
print("If all five sections above ran without errors, the runtime is good.")